# BMW Satış Verisi — Temizlik & Özellik Mühendisliği

Bu notebook, ham `BMW sales data (2010-2024).csv` dosyasını BigQuery'e ve Power BI'a
aktarılmaya hazır hale getirmek için gereken adımları içerir.

**Akış:** Veri kalitesi kontrolü → Aykırı değer tespiti → Özellik mühendisliği →
Kategorik standardizasyon → Temiz CSV çıktısı.


## 0. Kütüphaneler ve Veriyi Yükleme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Dosya yolunu kendi ortamına göre güncelle
# Kaggle'da: '/kaggle/input/bmw-sales-dataset/BMW sales data (2010-2024) (1).csv'
# Yerelde:   'BMW sales data (2010-2024).csv'
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])


Saving archive (1).zip to archive (1).zip


## 1. Veri Kalitesi Kontrolü

İlk iş: eksik değer var mı, tekrarlayan satır var mı, veri tipleri doğru mu?
Bu kontrolleri SQL'de tekrar yapmamak için burada halletmek daha hızlı.


In [ ]:
# Genel yapı
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model                 50000 non-null  object 
 1   Year                  50000 non-null  int64  
 2   Region                50000 non-null  object 
 3   Color                 50000 non-null  object 
 4   Fuel_Type             50000 non-null  object 
 5   Transmission          50000 non-null  object 
 6   Engine_Size_L         50000 non-null  float64
 7   Mileage_KM            50000 non-null  int64  
 8   Price_USD             50000 non-null  int64  
 9   Sales_Volume          50000 non-null  int64  
 10  Sales_Classification  50000 non-null  object 
dtypes: float64(1), int64(4), object(6)
memory usage: 4.2+ MB


In [ ]:
# Eksik değerler
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Eksik değer bulunan sütunlar:")
print(missing if len(missing) else "Yok — veri bu açıdan temiz.")


Eksik değer bulunan sütunlar:
Yok — veri bu açıdan temiz.


In [ ]:
# Tekrarlayan satırlar
dup_count = df.duplicated().sum()
print(f"Tekrarlayan satır sayısı: {dup_count}")

if dup_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Tekrarlar silindi. Yeni satır sayısı: {len(df)}")


Tekrarlayan satır sayısı: 0


In [ ]:
# Beklenmedik değerler var mı? (negatif fiyat, sıfır km vb.)
print(df[["Engine_Size_L", "Mileage_KM", "Price_USD", "Sales_Volume", "Year"]].describe())


       Engine_Size_L     Mileage_KM      Price_USD  Sales_Volume          Year
count   50000.000000   50000.000000   50000.000000  50000.000000  50000.000000
mean        3.247180  100307.203140   75034.600900   5067.514680   2017.015700
std         1.009078   57941.509344   25998.248882   2856.767125      4.324459
min         1.500000       3.000000   30000.000000    100.000000   2010.000000
25%         2.400000   50178.000000   52434.750000   2588.000000   2013.000000
50%         3.200000  100388.500000   75011.500000   5087.000000   2017.000000
75%         4.100000  150630.250000   97628.250000   7537.250000   2021.000000
max         5.000000  199996.000000  119998.000000   9999.000000   2024.000000


## 2. Aykırı Değer (Outlier) Tespiti

IQR (Interquartile Range) yöntemiyle sayısal sütunlardaki uç değerleri işaretliyoruz.
Burada **silmiyoruz** — çünkü bu gerçek satış verisi olabilir (örneğin çok yüksek
fiyatlı bir M Performance modeli aykırı değil, gerçek bir veri noktası olabilir).
Sadece flag'liyoruz ki Power BI'da isteğe göre filtrelenebilsin.


In [ ]:
def iqr_outlier_bounds(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

numeric_cols = ["Engine_Size_L", "Mileage_KM", "Price_USD", "Sales_Volume"]

for col in numeric_cols:
    low, high = iqr_outlier_bounds(df[col])
    flag_col = f"{col}_Outlier"
    df[flag_col] = ~df[col].between(low, high)
    print(f"{col}: sınırlar [{low:.1f}, {high:.1f}] — aykırı sayısı: {df[flag_col].sum()}")


Engine_Size_L: sınırlar [-0.1, 6.6] — aykırı sayısı: 0
Mileage_KM: sınırlar [-100500.4, 301308.6] — aykırı sayısı: 0
Price_USD: sınırlar [-15355.5, 165418.5] — aykırı sayısı: 0
Sales_Volume: sınırlar [-4835.9, 14961.1] — aykırı sayısı: 0


## 3. Özellik Mühendisliği (Feature Engineering)

LinkedIn gönderindeki dashboard'da geçen metrikleri destekleyecek yeni sütunlar:

- `Car_Age` — aracın kaç yaşında olduğu (Time Intelligence için)
- `Revenue` — Price_USD × Sales_Volume (Total Net Revenue ölçüsünün temeli)
- `Price_per_KM` — kilometre başına fiyat (kabaca amortisman göstergesi)
- `Series_Group` — Model adından üst kategori çıkarımı (X Series, i-Series, Core Sedans, M Performance)
- `Mileage_Category` — düşük/orta/yüksek km segmenti


In [ ]:
CURRENT_YEAR = 2024  # veri setinin son yılı; BigQuery'de de sabit tutulmalı

df["Car_Age"] = CURRENT_YEAR - df["Year"]
df["Revenue"] = df["Price_USD"] * df["Sales_Volume"]
df["Price_per_KM"] = df["Price_USD"] / df["Mileage_KM"].replace(0, np.nan)

df[["Car_Age", "Revenue", "Price_per_KM"]].describe()


,Car_Age,Revenue,Price_per_KM
count,50000.000000,5.000000e+04,50000.000000
mean,6.984300,3.802449e+08,4.114126
std,4.324459,2.623406e+08,145.390737
min,0.000000,3.410472e+06,0.152637
25%,3.000000,1.689871e+08,0.476764
50%,7.000000,3.318010e+08,0.748950
75%,11.000000,5.486494e+08,1.504195
max,14.000000,1.194346e+09,31311.000000


In [ ]:
def map_series_group(model: str) -> str:
    model = str(model).strip()
    if model.startswith("i"):
        return "i-Series (EV)"
    if model.startswith(("X", "M")):
        return "X Series / M Performance"
    if model in ["3 Series", "5 Series", "7 Series"]:
        return "Core Sedans"
    return "Other"

df["Series_Group"] = df["Model"].apply(map_series_group)
df["Series_Group"].value_counts()


,count
Series_Group,
X Series / M Performance,26923
Core Sedans,13853
i-Series (EV),9224


In [ ]:
bins = [0, 30000, 100000, np.inf]
labels = ["Low (0-30k km)", "Medium (30k-100k km)", "High (100k+ km)"]
df["Mileage_Category"] = pd.cut(df["Mileage_KM"], bins=bins, labels=labels)

df["Mileage_Category"].value_counts()


,count
Mileage_Category,
High (100k+ km),25085
Medium (30k-100k km),17356
Low (0-30k km),7559


## 4. Kategorik Değişkenleri Standartlaştırma

Metin sütunlarında boşluk/büyük-küçük harf tutarsızlığı BigQuery'de `GROUP BY`
sonuçlarını bozar (örn. "Petrol" ve "petrol " ayrı gruplar sayılır). Burada temizliyoruz.


In [ ]:
categorical_cols = ["Region", "Color", "Fuel_Type", "Transmission", "Sales_Classification"]

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

for col in categorical_cols:
    print(col, "->", sorted(df[col].unique()))


Region -> ['Africa', 'Asia', 'Europe', 'Middle East', 'North America', 'South America']
Color -> ['Black', 'Blue', 'Grey', 'Red', 'Silver', 'White']
Fuel_Type -> ['Diesel', 'Electric', 'Hybrid', 'Petrol']
Transmission -> ['Automatic', 'Manual']
Sales_Classification -> ['High', 'Low']


## 5. Temizlenmiş Veriyi Dışa Aktarma

Bu CSV artık BigQuery'e yüklenmeye hazır: eksik/tekrar kontrolü yapılmış,
aykırı değerler flag'lenmiş, yeni analiz sütunları eklenmiş.


In [ ]:
OUTPUT_PATH = "bmw_sales_cleaned.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Kaydedildi: {OUTPUT_PATH}")
print(f"Son boyut: {df.shape}")
df.head()


Kaydedildi: bmw_sales_cleaned.csv
Son boyut: (50000, 20)


,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification,Engine_Size_L_Outlier,Mileage_KM_Outlier,Price_USD_Outlier,Sales_Volume_Outlier,Car_Age,Revenue,Price_per_KM,Series_Group,Mileage_Category
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High,False,False,False,False,8,819542000,0.650684,Core Sedans,High (100k+ km)
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low,False,False,False,False,11,271562732,0.651092,i-Series (EV),High (100k+ km)
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low,False,False,False,False,2,792175410,10.305250,Core Sedans,Low (0-30k km)
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low,False,False,False,False,0,246749637,2.237057,X Series / M Performance,Low (0-30k km)
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low,False,False,False,False,4,153685840,0.408561,Core Sedans,High (100k+ km)


## Sıradaki Adım

Bu `bmw_sales_cleaned.csv` dosyasını BigQuery'e yükleyip bir önceki mesajda
konuştuğumuz SQL sorgularını (Region bazlı özet, EV/ICE karşılaştırması,
yıllık trend) bu temiz veri üzerinden çalıştırabilirsin. `Series_Group` sütunu
sayesinde artık SQL'de her seferinde CASE WHEN yazmana da gerek kalmadı.
